[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/05_matrix_factorization/05_proyecto_ials_bpr.ipynb)

# Proyecto 05 · «Recomendado para ti» v2: iALS + BPR en GPU, tuneados con Optuna
| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración** | 4–6 h |
| **GPU** | T4 (≈ 1–2 unidades con `FAST_DEV_RUN=False`) |
| **Prerrequisitos** | Lección 05, proyecto 04 |

## 🏢 Contexto de negocio
La fila «Recomendado para ti» de CineMatch usa EASE (proyecto 04). El equipo de plataforma quiere
pasar a **factores latentes** porque (1) los vectores se pueden indexar en un ANN y servir a escala,
(2) permiten **fold-in** de usuarios nuevos en tiempo real y (3) sirven como features del ranker.
Tu misión: entrenar **iALS** y **BPR** en GPU, tunearlos con **Optuna** con el mismo presupuesto, y
demostrar si igualan o superan a EASE.

## 📦 Dataset
MovieLens 1M, split temporal global (utilidades), validación temporal dentro de train. Implícito =
cualquier rating; relevante = rating ≥ 4.

## 📋 Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | `train_bpr()` en PyTorch con negativos uniformes y opción de **negativos por popularidad** | Corre en GPU; curva de pérdida y NDCG |
| 2 | Estudios Optuna para iALS y BPR (mismo nº de trials) | ≥ 20 trials cada uno (`FAST_DEV_RUN=False`: ≥ 40) |
| 3 | Tabla final en test (NDCG@10, Recall@10, Coverage) con Popularidad, EASE, iALS, BPR | **iALS ≥ 0,225** NDCG@10; **BPR ≥ 0,20** |
| 4 | Importancia de hiperparámetros (Optuna) + interpretación | 1 párrafo |
| 5 | `recommend_new_user(liked_titles)` con **fold-in** y su latencia | < 5 ms por usuario en CPU |

In [ ]:
!pip install -q implicit optuna pyarrow

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings("ignore")
seed = 42; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_RUN = True
N_TRIALS = 12 if FAST_DEV_RUN else 40
print("device:", device)

## 🧰 Utilidades del curso

Para que este notebook sea **autocontenido en Colab**, las tres celdas siguientes
contienen versiones mínimas de las utilidades que construimos en
[01_data](../01_data) (carga de MovieLens + **split temporal global**) y
[02_evaluation](../02_evaluation) (**NDCG@K, Recall@K, coverage**). Son las mismas
definiciones; si ya tienes tu propia librería de los módulos 01/02, puedes
importarla en su lugar.

Convenciones que usaremos en todo el Bloque II:

| Concepto | Convención |
|---|---|
| Columnas | `user_id`, `item_id`, `rating`, `timestamp` (IDs originales de MovieLens) |
| Índices internos | `uidx`, `iidx` contiguos `0..n-1`, definidos **solo con train** |
| Split | temporal **global**: el 80 % más antiguo de las interacciones → train, el 20 % más reciente → test |
| Implícito | positivo = `rating ≥ 4` (protocolo de Liang et al. 2018 / Steck 2019) |
| Evaluación | *full ranking* sobre todo el catálogo, excluyendo lo ya visto en train |

In [ ]:
# 🧰 Utilidades del curso (1/3) — carga de datos. Mismas funciones que en 01_data.
import io, os, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA_DIR = Path(os.environ.get("RECSYS_DATA", "data"))
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
# Espejo público en GitHub (mismo contenido, en CSV) por si GroupLens no responde
ML1M_MIRROR = "https://raw.githubusercontent.com/khanhnamle1994/movielens/master"


def load_movielens_1m(data_dir: Path = DATA_DIR):
    """Devuelve (ratings, movies, users) de MovieLens 1M, cacheados en parquet."""
    d = Path(data_dir) / "ml-1m"
    d.mkdir(parents=True, exist_ok=True)
    cache = {n: d / f"{n}.parquet" for n in ("ratings", "movies", "users")}
    if all(p.exists() for p in cache.values()):
        return tuple(pd.read_parquet(p) for p in cache.values())
    try:
        with urllib.request.urlopen(ML1M_URL, timeout=60) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(d.parent)
        rd = lambda f, cols: pd.read_csv(d / f, sep="::", engine="python", names=cols, encoding="latin-1")
        ratings = rd("ratings.dat", ["user_id", "item_id", "rating", "timestamp"])
        movies = rd("movies.dat", ["item_id", "title", "genres"])
        users = rd("users.dat", ["user_id", "gender", "age", "occupation", "zip"])
    except Exception as e:  # fallback: espejo en GitHub
        print(f"GroupLens no disponible ({e.__class__.__name__}); usando espejo de GitHub…")
        rd = lambda f: pd.read_csv(f"{ML1M_MIRROR}/{f}", sep="\t", index_col=0, encoding="latin-1")
        ratings = rd("ratings.csv").rename(columns={"movie_id": "item_id"})[["user_id", "item_id", "rating", "timestamp"]]
        movies = rd("movies.csv").rename(columns={"movie_id": "item_id"})[["item_id", "title", "genres"]]
        users = rd("users.csv").rename(columns={"zipcode": "zip"})[["user_id", "gender", "age", "occupation", "zip"]]
    movies["genres"] = movies["genres"].str.split("|")
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$").astype(float)
    for n, df in zip(cache, (ratings, movies, users)):
        df.to_parquet(cache[n], index=False)
    return ratings, movies, users

In [ ]:
# 🧰 Utilidades del curso (2/3) — split temporal y codificación. Mismas funciones que en 01_data.
def temporal_split(df: pd.DataFrame, test_frac: float = 0.2, ts_col: str = "timestamp"):
    """Split temporal GLOBAL: todo lo anterior al corte → train; lo posterior → test.
    El test se filtra a usuarios e ítems vistos en train (warm-start)."""
    cutoff = df[ts_col].quantile(1 - test_frac)
    train, test = df[df[ts_col] <= cutoff], df[df[ts_col] > cutoff]
    test = test[test.user_id.isin(train.user_id) & test.item_id.isin(train.item_id)]
    return train.reset_index(drop=True), test.reset_index(drop=True)


class Encoder:
    """Mapea IDs originales ↔ índices contiguos (ajustado SOLO con train)."""
    def __init__(self, train: pd.DataFrame):
        self.users = np.sort(train.user_id.unique())
        self.items = np.sort(train.item_id.unique())
        self.u2i = pd.Series(np.arange(len(self.users)), index=self.users)
        self.i2i = pd.Series(np.arange(len(self.items)), index=self.items)
        self.n_users, self.n_items = len(self.users), len(self.items)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df[df.user_id.isin(self.u2i.index) & df.item_id.isin(self.i2i.index)].copy()
        df["uidx"] = self.u2i.loc[df.user_id].values
        df["iidx"] = self.i2i.loc[df.item_id].values
        return df

    def csr(self, df: pd.DataFrame, values=None) -> sp.csr_matrix:
        """Matriz usuario×ítem dispersa (binaria por defecto)."""
        v = np.ones(len(df), dtype=np.float32) if values is None else np.asarray(values, dtype=np.float32)
        return sp.csr_matrix((v, (df.uidx.values, df.iidx.values)), shape=(self.n_users, self.n_items))

In [ ]:
# 🧰 Utilidades del curso (3/3) — métricas top-K. Mismas definiciones que en 02_evaluation.
from typing import Callable, Dict


def test_relevance(test_enc: pd.DataFrame, min_rating: float = 4.0) -> Dict[int, np.ndarray]:
    """uidx → array de iidx relevantes en test (rating ≥ min_rating)."""
    rel = test_enc[test_enc.rating >= min_rating]
    return {u: g.values for u, g in rel.groupby("uidx")["iidx"]}


def topk_from_scores(scores: np.ndarray, seen: sp.csr_matrix, k: int) -> np.ndarray:
    """Top-K por fila excluyendo ítems ya vistos (scores: batch×n_items, seen: batch×n_items)."""
    scores = np.array(scores, dtype=np.float32, copy=True)
    r, c = seen.nonzero()
    scores[r, c] = -np.inf
    part = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.take_along_axis(scores, part, 1).argsort(axis=1)[:, ::-1]
    return np.take_along_axis(part, order, 1)


def ndcg_recall_at_k(recs: np.ndarray, relevant: list, k: int):
    """NDCG@K y Recall@K (binarios) para cada usuario."""
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ndcg, recall = np.zeros(len(recs)), np.zeros(len(recs))
    for n, (rec, rel) in enumerate(zip(recs, relevant)):
        hits = np.isin(rec[:k], rel)
        idcg = discounts[: min(len(rel), k)].sum()
        ndcg[n] = (hits * discounts).sum() / idcg
        recall[n] = hits.sum() / len(rel)
    return ndcg, recall


def evaluate_topk(score_fn: Callable[[np.ndarray], np.ndarray], train_csr: sp.csr_matrix,
                  relevance: Dict[int, np.ndarray], k: int = 10, batch: int = 512) -> Dict[str, float]:
    """Evalúa un modelo dado por score_fn(uidx_batch) -> scores densos (batch×n_items)."""
    users = np.array(sorted(relevance))
    all_recs = []
    for s in range(0, len(users), batch):
        ub = users[s : s + batch]
        all_recs.append(topk_from_scores(score_fn(ub), train_csr[ub], k))
    recs = np.vstack(all_recs)
    ndcg, rec = ndcg_recall_at_k(recs, [relevance[u] for u in users], k)
    return {f"NDCG@{k}": ndcg.mean(), f"Recall@{k}": rec.mean(),
            "Coverage": len(np.unique(recs)) / train_csr.shape[1], "n_users": len(users)}

In [ ]:
ratings, movies, users = load_movielens_1m()
train, test = temporal_split(ratings)
enc = Encoder(train); X = enc.csr(enc.transform(train)); rel = test_relevance(enc.transform(test))
tr_in, val = temporal_split(train, test_frac=0.1)
enc_v = Encoder(tr_in); Xv = enc_v.csr(enc_v.transform(tr_in)); relv = test_relevance(enc_v.transform(val))
titles = movies.set_index("item_id").title.loc[enc.items].values
pop = np.asarray(X.sum(0)).ravel().astype(np.float32)

In [ ]:
# iALS con la librería `implicit` (GPU si hay CUDA; si no, CPU multihilo)
import implicit
from implicit.als import AlternatingLeastSquares

USE_GPU = bool(getattr(implicit.gpu, "HAS_CUDA", False)) and torch.cuda.is_available()


def fit_ials(X: sp.csr_matrix, factors=128, reg=50.0, alpha=1.0, iters=15):
    m = AlternatingLeastSquares(factors=factors, regularization=reg, alpha=alpha, iterations=iters,
                                use_gpu=USE_GPU, random_state=seed)
    m.fit(X, show_progress=False)
    if USE_GPU:
        m = m.to_cpu()                       # factores como numpy para evaluar
    return m


def mf_score_fn(U: np.ndarray, V: np.ndarray):
    return lambda ub: U[ub] @ V.T

## Paso 1 · BPR en PyTorch  ✏️ TODO
Implementa `train_bpr(X, k, lr, reg, epochs, neg="uniform"|"pop", pop_power=0.75)`:
- Modelo: embeddings de usuario/ítem + sesgo de ítem; pérdida $-\log\sigma(\hat x_{ui}-\hat x_{uj})$ + L2.
- `neg="pop"`: muestrea $j \propto n_j^{0.75}$ (como word2vec) con `torch.multinomial`.
- Devuelve el modelo y una función `score_fn(ub)` compatible con `evaluate_topk`.

In [ ]:
def train_bpr(X, k=64, lr=5e-3, reg=1e-5, epochs=20, batch=8192, neg="uniform", pop_power=0.75):
    # TODO
    raise NotImplementedError

## Paso 2 · Optuna  ✏️ TODO
Define `objective_ials(trial)` (factors ∈ {32…512}, regularization log ∈ [0,1; 1 000], alpha log ∈ [0,1; 50],
iterations ∈ [5, 30]) y `objective_bpr(trial)` (k, lr, reg, epochs, neg). Maximiza NDCG@10 en validación
con `TPESampler(seed=42)`. Usa el mismo `N_TRIALS` para ambos (comparación justa).

In [ ]:
# TODO: estudios de Optuna

## Paso 3 · Evaluación final y comparación con EASE  ✏️ TODO
Re-entrena con todo `X` usando los mejores hiperparámetros. Añade Popularidad y EASE (λ en validación).

In [ ]:
# TODO: tabla final

## Paso 4 · Fold-in para usuarios nuevos  ✏️ TODO
`recommend_new_user(liked_titles, k=10)` usando los factores de ítem del mejor iALS. Mide la latencia.

In [ ]:
# TODO: fold-in

---
# ⛔ SPOILER — intenta resolverlo primero
Solución de referencia completa.

In [ ]:
import torch.nn as nn
import torch.nn.functional as Fnn


class BPRMF(nn.Module):
    def __init__(self, n_users, n_items, k):
        super().__init__()
        self.P, self.Q, self.b = nn.Embedding(n_users, k), nn.Embedding(n_items, k), nn.Embedding(n_items, 1)
        nn.init.normal_(self.P.weight, std=0.01); nn.init.normal_(self.Q.weight, std=0.01); nn.init.zeros_(self.b.weight)

    def forward(self, u, i, j):
        return (self.P(u) * (self.Q(i) - self.Q(j))).sum(1) + (self.b(i) - self.b(j)).squeeze(1)


def train_bpr(X, k=64, lr=5e-3, reg=1e-5, epochs=20, batch=8192, neg="uniform", pop_power=0.75, return_hist=False):
    torch.manual_seed(seed)
    model = BPRMF(*X.shape, k).to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    uu, ii = X.nonzero(); uu, ii = torch.tensor(uu, device=device), torch.tensor(ii, device=device)
    probs = torch.tensor(np.asarray(X.sum(0)).ravel() ** pop_power, dtype=torch.float32, device=device)
    hist = []
    for ep in range(epochs):
        perm, tot = torch.randperm(len(uu), device=device), 0.0
        for s in range(0, len(uu), batch):
            b = perm[s:s + batch]; u, i = uu[b], ii[b]
            j = (torch.randint(0, X.shape[1], (len(b),), device=device) if neg == "uniform"
                 else torch.multinomial(probs, len(b), replacement=True))
            l2 = model.P(u).pow(2).sum() + model.Q(i).pow(2).sum() + model.Q(j).pow(2).sum()
            loss = -Fnn.logsigmoid(model(u, i, j)).mean() + reg * l2 / len(b)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
        hist.append(tot / len(uu))
    P = model.P.weight.detach().cpu().numpy(); Q = model.Q.weight.detach().cpu().numpy(); bb = model.b.weight.detach().cpu().numpy().ravel()
    score_fn = lambda ub: P[ub] @ Q.T + bb
    return (model, score_fn, hist) if return_hist else (model, score_fn)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_ials(trial):
    factor_grid = [32, 64, 128] if FAST_DEV_RUN else [32, 64, 128, 256, 512]
    m = fit_ials(Xv, factors=trial.suggest_categorical("factors", factor_grid),
                 reg=trial.suggest_float("reg", 0.1, 1000.0, log=True), alpha=trial.suggest_float("alpha", 0.1, 50.0, log=True),
                 iters=trial.suggest_int("iters", 5, 20 if FAST_DEV_RUN else 30))
    return evaluate_topk(mf_score_fn(m.user_factors, m.item_factors), Xv, relv)["NDCG@10"]


def objective_bpr(trial):
    _, f = train_bpr(Xv, k=trial.suggest_categorical("k", [32, 64, 128]), lr=trial.suggest_float("lr", 1e-3, 3e-2, log=True),
                     reg=trial.suggest_float("reg", 1e-7, 1e-3, log=True), epochs=trial.suggest_int("epochs", 5, 30 if FAST_DEV_RUN else 60),
                     neg=trial.suggest_categorical("neg", ["uniform", "pop"]))
    return evaluate_topk(f, Xv, relv)["NDCG@10"]


studies = {}
for name, obj in [("iALS", objective_ials), ("BPR", objective_bpr)]:
    t0 = time.time()
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    st.optimize(obj, n_trials=N_TRIALS)
    studies[name] = st
    print(f"{name}: mejor NDCG@10 val = {st.best_value:.4f} · {st.best_params} · {time.time() - t0:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for name, st in studies.items():
    v = [t.value for t in st.trials]
    axes[0].plot(np.maximum.accumulate(v), "o-", label=name)
axes[0].set_xlabel("trial"); axes[0].set_ylabel("mejor NDCG@10 val"); axes[0].set_title("Historial de optimización"); axes[0].legend()
for ax, (name, st) in zip(axes[1:], studies.items()):
    try:
        imp = optuna.importance.get_param_importances(st)
        ax.barh(list(imp)[::-1], list(imp.values())[::-1], color="#4c72b0"); ax.set_title(f"Importancia de hiperparámetros · {name}")
    except Exception as e:
        ax.text(0.1, 0.5, f"importancia no disponible: {e.__class__.__name__}"); ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
def ease(Xm, lam):
    G = torch.tensor((Xm.T @ Xm).toarray(), dtype=torch.float32, device=device)
    P = torch.linalg.inv(G + lam * torch.eye(G.shape[0], device=device)); B = -P / torch.diag(P); B.fill_diagonal_(0)
    return B.cpu().numpy()


lam_b = max([500, 1000, 3000, 10000], key=lambda l: evaluate_topk(lambda ub: np.asarray(Xv[ub] @ ease(Xv, l)), Xv, relv)["NDCG@10"])
B = ease(X, lam_b)
bi = studies["iALS"].best_params
ials = fit_ials(X, factors=bi["factors"], reg=bi["reg"], alpha=bi["alpha"], iters=bi["iters"])
bb = studies["BPR"].best_params
bpr_model, bpr_fn, bpr_hist = train_bpr(X, k=bb["k"], lr=bb["lr"], reg=bb["reg"], epochs=bb["epochs"], neg=bb["neg"], return_hist=True)
final = {"Popularidad": evaluate_topk(lambda ub: np.tile(pop, (len(ub), 1)), X, rel),
         f"EASE (λ={lam_b})": evaluate_topk(lambda ub: np.asarray(X[ub] @ B), X, rel),
         "iALS (Optuna)": evaluate_topk(mf_score_fn(ials.user_factors, ials.item_factors), X, rel),
         "BPR (Optuna)": evaluate_topk(bpr_fn, X, rel)}
final_df = pd.DataFrame(final).T.drop(columns="n_users")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
final_df["NDCG@10"].plot.barh(ax=axes[0], color="#4c72b0"); axes[0].set_xlim(final_df["NDCG@10"].min() * 0.9, None); axes[0].set_title("NDCG@10 test")
axes[1].plot(bpr_hist, "o-"); axes[1].set_xlabel("época"); axes[1].set_ylabel("pérdida BPR"); axes[1].set_title("Entrenamiento final de BPR")
plt.tight_layout(); plt.show()
final_df.round(4)

In [ ]:
V_items = ials.item_factors
lam_fold = bi["reg"]; alpha_fold = bi["alpha"]
VtV = V_items.T @ V_items + lam_fold * np.eye(V_items.shape[1])      # precomputable una vez


def recommend_new_user(liked_titles, k=10):
    idx = [int(np.flatnonzero(pd.Series(titles).str.contains(t, regex=False))[0]) for t in liked_titles]
    Vu = V_items[idx]; cu = np.full(len(idx), alpha_fold)
    # implicit usa confianza c = α·r en las celdas observadas (sin el +1 de Hu et al.); replicamos su ecuación
    u = np.linalg.solve(VtV + (Vu.T * (cu - 1)) @ Vu, Vu.T @ cu)
    s = V_items @ u; s[idx] = -np.inf
    return list(titles[np.argsort(-s)[:k]])


t0 = time.time()
for _ in range(100):
    recs = recommend_new_user(["Alien (1979)", "Terminator, The (1984)", "Blade Runner (1982)"])
print(f"Latencia fold-in + scoring: {(time.time() - t0) * 10:.2f} ms/usuario")
print("Fan de la ciencia ficción ochentera →", recs[:8])
print("Familia con niños →", recommend_new_user(["Toy Story (1995)", "Lion King, The (1994)", "Aladdin (1992)"])[:8])

### 📝 Interpretación (solución de referencia)
- En ML-1M temporal, la **regularización** de iALS es con diferencia el hiperparámetro más importante:
  valores altos empujan hacia lo popular (que en el futuro paga) y valores bajos sobreajustan al pasado.
- BPR con negativos por popularidad suele converger más rápido que con negativos uniformes y es más
  sensible al *learning rate*; con el mismo presupuesto de trials suele quedar por detrás de iALS.
- iALS iguala o supera a EASE con vectores de 64–512 dimensiones que **sí** se pueden indexar en un ANN
  y servir con fold-in: es la opción para producción a escala.

## 🚀 Retos extra (nivel experto)
1. **iALS revisitado**: implementa la regularización escalada por frecuencia de Rendle et al. (2022) en
   tu iALS desde cero y comprueba si mejora con dimensiones grandes (512–2048).
2. **WARP** con LightFM (`pip install lightfm`; puede requerir compilar) o implementa WARP en PyTorch
   (muestrear negativos hasta violar el margen). Compara precision@5 con BPR.
3. **Fold-in con implicit**: usa `model.recommend(..., recalculate_user=True)` y compara con tu versión.
4. **Ensemble** EASE + iALS (combinación lineal de scores z-normalizados con peso en validación).
5. Indexa los vectores de ítem en **FAISS** (`IndexFlatIP`) y mide recall@100 del ANN vs. scoring exacto.

## 🤔 Reflexión (producción / MLOps)
- ¿Qué artefactos versionarías en MLflow (factores, mapeos de IDs, hiperparámetros, estudio de Optuna)?
- Los vectores de usuario se quedan obsoletos: ¿re-entreno nocturno, fold-in online o ambos?
- ¿Cómo detectarías *embedding drift* entre dos re-entrenos (pista: alinear con Procrustes o comparar vecinos)?